# Predição do modelo de classificação de Coral-Sol utilizando imagens Prova de Fogo da PUC

Autor:  Matheus Santos

Atualizado: 19/02/2026

## Objetivo:
Notebook destinado à predição das imagens das provas de foto utilizando o modelo V4 de classificação de Coral-Sol, com o objetivo de analisar seu desempenho.

## Importações Necessárias

In [ ]:
!pip install numpy opencv-python matplotlib ultralytics scikit-learn nbconvert

In [16]:
import numpy as np
import pandas as pd
import cv2
import glob
from pathlib import Path
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

## Declaração de Constantes e Modelos

In [17]:
model_v1 = YOLO('./coral_yolov11_class_8c_V4.pt')

## Funções Necessárias

In [33]:
def read_image(filename: str) -> np.ndarray:
    """
    Load an image from disk, convert it to RGB format, and perform a centered
    vertical crop while preserving the original width.

    The crop keeps approximately 68% of the image height (34% above and
    34% below the vertical center).

    Args:
        filename (str): Path to the image file.

    Returns:
        np.ndarray: Cropped RGB image as uint8 array.
        None: If the image cannot be loaded or an error occurs.
    """
    try:
        # Read image using OpenCV (BGR format by default)
        image = cv2.imread(filename)

        # Check if the image was successfully loaded
        if image is None:
            print(f"Error loading image: {filename}")
            return None
        
        # Convert from BGR (OpenCV default) to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Get image dimensions
        height, width, _ = image.shape

        # Compute vertical center
        mid = height // 2

        # Define top and bottom crop boundaries (34% above and below center)
        top = max(0, mid - int(0.34 * height))
        bottom = min(height, mid + int(0.34 * height))

        # Perform vertical central crop (full width preserved)
        cropped_image = image[top:bottom, :]

        # Ensure output type is uint8
        return cropped_image.astype(np.uint8)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None


def plot_results(
    image: np.ndarray,
    image_name: str,
    coral_level: str,
    true_label: str,
    predicted_label_v1: str,
    v1_score: float
) -> None:
    """
    Display the input image alongside ground-truth and model prediction results.

    Two panels are shown:
    - Left: Ground-truth label
    - Right: Model prediction with confidence score

    Args:
        image (np.ndarray): Image array in RGB format.
        image_name (str): Name or identifier of the image.
        coral_level (str): Coral infestation level metadata.
        true_label (str): Ground-truth classification label.
        predicted_label_v1 (str): Predicted label from Model V1.
        v1_score (float): Confidence score from Model V1.
    """

    # Create side-by-side subplot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Global figure title
    fig.suptitle(
        f"Image: {image_name} | Coral-Sol Level: {coral_level}",
        fontsize=12,
        fontweight="bold"
    )

    # Left panel: ground truth
    axes[0].imshow(image)
    axes[0].set_title(f"True Label: {true_label}", fontsize=12)
    axes[0].axis('off')

    # Right panel: model prediction
    axes[1].imshow(image)
    axes[1].set_title(
        f"Model Prediction: {predicted_label_v1} ({v1_score:.2f})",
        fontsize=12
    )
    axes[1].axis('off')

    # Adjust layout and display
    plt.tight_layout()
    plt.show()


def evaluate_classification(true_labels: list[str], predicted_labels_v1: list[str]) -> None:
    """
    Evaluate binary classification performance for Model V1.

    This function:
    - Converts string labels ("Positive"/"Negative") to binary format (1/0)
    - Computes and displays the confusion matrix
    - Prints the classification report (precision, recall, F1-score)

    Args:
        true_labels (list[str]): Ground-truth labels as strings.
        predicted_labels_v1 (list[str]): Predicted labels from Model V1.
    """

    # Convert string labels to binary format (Positive=1, Negative=0)
    true_labels_binary = [1 if label == 'Positive' else 0 for label in true_labels]
    predicted_labels_binary_v1 = [
        1 if label == 'Positive' else 0 for label in predicted_labels_v1
    ]

    # Force confusion matrix to always display both classes [0, 1]
    conf_matrix_v1 = confusion_matrix(
        true_labels_binary,
        predicted_labels_binary_v1,
        labels=[0, 1]
    )

    # Create confusion matrix display object
    disp_v1 = ConfusionMatrixDisplay(
        confusion_matrix=conf_matrix_v1,
        display_labels=['Negative', 'Positive']
    )

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    disp_v1.plot(cmap=plt.cm.Blues, ax=ax, values_format="d")
    ax.set_title("Confusion Matrix - V1 Model")

    plt.tight_layout()
    plt.show()

    # Print classification metrics
    print("Classification Report - V1 Model:")
    print(
        classification_report(
            true_labels_binary,
            predicted_labels_binary_v1,
            labels=[0, 1],
            target_names=['Negative', 'Positive'],
            zero_division=0  # Avoid division-by-zero warnings
        )
    )

def evaluate_directory(
    input_directory: str,
    model_v1,
    nivel_coralsol_dict: dict
) -> tuple[list[str], list[str]]:
    """
    Evaluate a directory of images using Model V1.

    This function:
    - Recursively scans the input directory
    - Loads and preprocesses images
    - Performs model inference
    - Extracts ground-truth labels from folder structure
    - Stores predicted and true labels
    - Displays qualitative results for each image

    Expected folder structure:
        input_directory/
            Positivo/
                image1.jpg
            Negativo/
                image2.jpg

    Args:
        input_directory (str): Root directory containing images.
        model_v1: Trained classification model with a `.predict()` method.
        nivel_coralsol_dict (dict): Dictionary mapping image stem name
                                    to coral infestation level metadata.

    Returns:
        tuple[list[str], list[str]]:
            - true_labels
            - predicted_labels_v1
    """

    # Lists to store ground-truth and predicted labels
    true_labels = []
    predicted_labels_v1 = []

    # Recursively iterate through all files in the directory
    for filename in glob.glob(f"{input_directory}/**/*.*", recursive=True):
        try:
            # Skip invalid paths
            if not os.path.isfile(filename):
                continue

            # Load and preprocess image (RGB conversion + central crop)
            image = read_image(filename)
            if image is None:
                continue

            # ==================================================
            # MODEL PREDICTION (Model V1)
            # ==================================================
            # Run inference
            results_v1 = model_v1.predict(filename, verbose=False)

            # Extract class probabilities as NumPy array
            class_probabilities_v1 = results_v1[0].probs.data.cpu().numpy()

            # Get predicted class (index with highest probability)
            predicted_label_index_v1 = np.argmax(class_probabilities_v1)

            # Confidence score of predicted class
            v1_score = class_probabilities_v1[predicted_label_index_v1]

            # Convert numeric prediction to string label
            predicted_label_v1 = (
                "Positive" if predicted_label_index_v1 == 1 else "Negative"
            )

            # ==================================================
            # GROUND-TRUTH LABEL (BASED ON FOLDER STRUCTURE)
            # ==================================================
            path_obj = Path(filename)

            # Extract filename and filename without extension
            image_name = path_obj.name
            image_stem = path_obj.stem

            # Retrieve coral infestation level from metadata dictionary
            coral_level = nivel_coralsol_dict.get(image_stem, "Not informed")

            # Immediate parent folder defines the true class
            parent_folder = path_obj.parent.name

            if parent_folder.lower() == "positivo":
                true_label = "Positive"
            elif parent_folder.lower() == "negativo":
                true_label = "Negative"
            else:
                print(f"Unknown parent folder for {filename}")
                continue

            # ==================================================
            # STORE RESULTS
            # ==================================================
            true_labels.append(true_label)
            predicted_labels_v1.append(predicted_label_v1)

            # Display qualitative result
            plot_results(
                image=image,
                image_name=image_name,
                coral_level=coral_level,
                true_label=true_label,
                predicted_label_v1=predicted_label_v1,
                v1_score=v1_score
            )

        except Exception as e:
            # Prevent full evaluation from stopping due to a single error
            print(f"Error processing file {filename}: {e}")
            continue

    return true_labels, predicted_labels_v1



In [19]:
# Path to the Excel spreadsheet containing dataset metadata
PLANILHA_PATH = "./Nível de Dificuldade - Dataset PUC.xlsx"

# Load the spreadsheet into a pandas DataFrame
df = pd.read_excel(PLANILHA_PATH)

# Remove potential leading/trailing whitespace from the filename column
# This prevents key mismatches when accessing the dictionary later
df["Nome do arquivo"] = df["Nome do arquivo"].astype(str).str.strip()

# Create a dictionary mapping:
# filename → coral infestation level
# This allows fast lookup of coral level by image filename
nivel_coralsol_dict = dict(
    zip(df["Nome do arquivo"], df["Nivel do Coral-Sol"])
)


## Processamento das imagens - Fácil(Coluna Dificuldade da Imagem)

In [ ]:
true_labels, predicted_labels = evaluate_directory(
    input_directory="dataset_filtrado/Fácil",
    model_v1=model_v1,
    nivel_coralsol_dict=nivel_coralsol_dict
)


## Matriz de confusão - Fácil(Coluna Dificuldade da Imagem)

In [ ]:
evaluate_classification(true_labels, predicted_labels)

## Processamento das imagens - Médio(Coluna Dificuldade da Imagem)

In [ ]:
true_labels, predicted_labels = evaluate_directory(
    input_directory="dataset_filtrado/Médio",
    model_v1=model_v1,
    nivel_coralsol_dict=nivel_coralsol_dict
)

## Matriz de confusão - Médio(Coluna Dificuldade da Imagem)

In [ ]:
evaluate_classification(true_labels, predicted_labels)

## Processamento das imagens - Difícil(Coluna Dificuldade da Imagem)

In [ ]:
true_labels, predicted_labels = evaluate_directory(
    input_directory="dataset_filtrado/Difícil",
    model_v1=model_v1,
    nivel_coralsol_dict=nivel_coralsol_dict
)

## Matriz de confusão - Difícil(Coluna Dificuldade da Imagem)

In [ ]:
evaluate_classification(true_labels, predicted_labels)

## Processamento das imagens Total

In [ ]:
true_labels, predicted_labels = evaluate_directory(
    input_directory="dataset_filtrado/Total",
    model_v1=model_v1,
    nivel_coralsol_dict=nivel_coralsol_dict
)

## Matriz de confusão - Total

In [ ]:
evaluate_classification(true_labels, predicted_labels)

In [ ]:
!jupyter nbconvert --to html prediction_prova_foto-comparacao.ipynb